In [24]:
import pandas as pd
import numpy as np

# Load the corpus
df = pd.read_json("released_data.json", lines=True)

# Basic check
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (7775, 8)
Columns: ['source', 'title', 'event_id', 'adfontes_fair', 'adfontes_political', 'allsides_bias', 'content', 'misc']


,source,title,event_id,adfontes_fair,adfontes_political,allsides_bias,content,misc
0,Fox News,"Trump blasts Howard Schultz, says ex-Starbucks...",0,bias,bias,From the Right,Obama administration alum Roger Fisk and Repub...,"{'time': '2019-01-28 16:10:44.680484', 'topics..."
1,USA TODAY,Trump blasts former Starbucks CEO Howard Schul...,0,bias,neutral,From the Center,WASHINGTON – President Donald Trump took a swi...,"{'time': 'None', 'topics': 'Election: Presiden..."
2,Washington Times,Mick Mulvaney: Trump to secure border 'with or...,0,bias,neutral,From the Right,Acting White House chief of staff Mick Mulvane...,"{'time': 'None', 'topics': 'White House', 'aut..."
3,Washington Times,Trump says 'we'll do the emergency' if border ...,0,bias,neutral,From the Right,President Trump repeated his vow Friday to dec...,"{'time': 'None', 'topics': 'White House, Polit..."
4,BBC News,Trump backs down to end painful shutdown tempo...,0,bias,neutral,From the Center,President Donald Trump has yielded to politica...,"{'time': '2019-01-26 00:00:00', 'topics': 'Whi..."


In [25]:
# Extract the full topic string from misc
df["topics"] = df["misc"].apply(
    lambda x: x.get("topics") if isinstance(x, dict) else None
)

# Calculate article length
df["word_count"] = df["content"].astype(str).str.split().str.len()

# Keep only rows usable for validation sampling
sampling_df = df[
    df["content"].notna() &
    (df["content"].astype(str).str.strip() != "") &
    df["source"].notna() &
    (df["source"].astype(str).str.strip() != "") &
    df["allsides_bias"].isin(["From the Left", "From the Center", "From the Right"]) &
    df["topics"].notna() &
    (df["topics"].astype(str).str.strip() != "")
].copy()

print("Sampling pool shape:", sampling_df.shape)
sampling_df.head()

Sampling pool shape: (7735, 10)


,source,title,event_id,adfontes_fair,adfontes_political,allsides_bias,content,misc,topics,word_count
0,Fox News,"Trump blasts Howard Schultz, says ex-Starbucks...",0,bias,bias,From the Right,Obama administration alum Roger Fisk and Repub...,"{'time': '2019-01-28 16:10:44.680484', 'topics...",Election: Presidential,688
1,USA TODAY,Trump blasts former Starbucks CEO Howard Schul...,0,bias,neutral,From the Center,WASHINGTON – President Donald Trump took a swi...,"{'time': 'None', 'topics': 'Election: Presiden...",Election: Presidential,397
2,Washington Times,Mick Mulvaney: Trump to secure border 'with or...,0,bias,neutral,From the Right,Acting White House chief of staff Mick Mulvane...,"{'time': 'None', 'topics': 'White House', 'aut...",White House,405
3,Washington Times,Trump says 'we'll do the emergency' if border ...,0,bias,neutral,From the Right,President Trump repeated his vow Friday to dec...,"{'time': 'None', 'topics': 'White House, Polit...","White House, Politics",183
4,BBC News,Trump backs down to end painful shutdown tempo...,0,bias,neutral,From the Center,President Donald Trump has yielded to politica...,"{'time': '2019-01-26 00:00:00', 'topics': 'Whi...","White House, Politics",1209


In [26]:
# Political orientation distribution
sampling_df["allsides_bias"].value_counts()

allsides_bias
From the Left      3663
From the Right     2839
From the Center    1233
Name: count, dtype: int64

In [27]:
# Topic distribution, keeping full topic strings
sampling_df["topics"].value_counts().head(20)

topics
Election: Presidential        1018
Politics                       548
White House                    511
Immigration                    360
Middle East                    305
Elections                      243
Healthcare                     238
US Senate                      196
World                          192
Violence in America            186
Supreme Court                  185
Terrorism                      184
Budget - Debt                  183
Gun Control and Gun Rights     163
US House                       135
Economy and Jobs               128
Media Bias                     127
National Security               99
North Korea                     97
Foreign Policy                  90
Name: count, dtype: int64

In [28]:
# Outlet distribution
sampling_df["source"].value_counts().head(20)

source
CNN (Web News)               1015
Fox News                     1001
New York Times - News         777
Washington Times              654
HuffPost                      534
Politico                      428
USA TODAY                     351
Townhall                      295
The Hill                      288
Reuters                       191
Newsmax                       182
Vox                           164
Washington Examiner           157
BBC News                      135
Breitbart News                116
Christian Science Monitor     101
ABC News                       97
CBN                            88
The Guardian                   84
TheBlaze.com                   83
Name: count, dtype: int64

In [29]:
# Article length summary
sampling_df["word_count"].describe()

count     7735.000000
mean       819.816677
std        942.587135
min          1.000000
25%        452.000000
50%        732.000000
75%       1058.000000
max      69312.000000
Name: word_count, dtype: float64

In [30]:
mean_length = sampling_df["word_count"].mean()

length_lower = 700
length_upper = 950

print("Mean word count:", round(mean_length, 2))
print("Length window:", length_lower, "-", length_upper)

Mean word count: 819.82
Length window: 700 - 950


In [31]:
length_filtered_df = sampling_df[
    sampling_df["word_count"].between(length_lower, length_upper)
].copy()

print("Length-filtered pool shape:", length_filtered_df.shape)

length_filtered_df["allsides_bias"].value_counts()

Length-filtered pool shape: (1609, 10)


allsides_bias
From the Left      699
From the Right     602
From the Center    308
Name: count, dtype: int64

In [32]:
# Topic distribution after length filtering
length_filtered_df["topics"].value_counts().head(30)

topics
Election: Presidential        219
Politics                      104
White House                   100
Immigration                    77
Middle East                    63
Healthcare                     61
Elections                      60
US Senate                      51
World                          46
Budget - Debt                  43
Terrorism                      43
Violence in America            41
Supreme Court                  36
US House                       34
Gun Control and Gun Rights     32
General News                   26
Economy and Jobs               23
Media Bias                     22
Trade                          21
US Congress                    20
Fiscal Cliff                   18
Justice Department             18
North Korea                    17
Environment                    17
Defense                        16
National Security              15
Foreign Policy                 15
FBI                            13
LGBT Rights                    13
US Mili

In [33]:
# Topic distribution by political orientation after length filtering
topic_orientation_counts = pd.crosstab(
    length_filtered_df["topics"],
    length_filtered_df["allsides_bias"]
)

topic_orientation_counts["total"] = topic_orientation_counts.sum(axis=1)

topic_orientation_counts = topic_orientation_counts.sort_values(
    "total",
    ascending=False
)

topic_orientation_counts.head(30)

allsides_bias,From the Center,From the Left,From the Right,total
topics,,,,
Election: Presidential,18,108,93,219
Politics,22,43,39,104
White House,26,38,36,100
Immigration,16,38,23,77
Middle East,14,25,24,63
Healthcare,9,31,21,61
Elections,15,26,19,60
US Senate,10,22,19,51
World,13,16,17,46


In [34]:
from collections import Counter

target_per_orientation = 15
max_per_outlet_total = 3
max_per_topic_total = 4

orientations = ["From the Left", "From the Center", "From the Right"]

selected_rows = []

outlet_counter = Counter()
topic_counter = Counter()

for orientation in orientations:
    orientation_pool = length_filtered_df[
        length_filtered_df["allsides_bias"] == orientation
    ].copy()
    
    # Distance from corpus mean length
    orientation_pool["length_distance"] = (
        orientation_pool["word_count"] - mean_length
    ).abs()
    
    # Sort by topic frequency first to keep common topics available,
    # then by closeness to mean length
    orientation_pool["topic_frequency"] = orientation_pool["topics"].map(
        length_filtered_df["topics"].value_counts()
    )
    
    orientation_pool = orientation_pool.sort_values(
        ["length_distance", "topic_frequency"],
        ascending=[True, False]
    )
    
    selected_for_orientation = []
    
    # First pass: enforce outlet/topic diversity strictly
    for idx, row in orientation_pool.iterrows():
        source = row["source"]
        topic = row["topics"]
        
        if outlet_counter[source] >= max_per_outlet_total:
            continue
        
        if topic_counter[topic] >= max_per_topic_total:
            continue
        
        selected_for_orientation.append(idx)
        outlet_counter[source] += 1
        topic_counter[topic] += 1
        
        if len(selected_for_orientation) == target_per_orientation:
            break
    
    # Fallback pass: if too few selected, relax topic constraint but keep outlet constraint
    if len(selected_for_orientation) < target_per_orientation:
        remaining_needed = target_per_orientation - len(selected_for_orientation)
        
        remaining_pool = orientation_pool[
            ~orientation_pool.index.isin(selected_for_orientation)
        ]
        
        for idx, row in remaining_pool.iterrows():
            source = row["source"]
            topic = row["topics"]
            
            if outlet_counter[source] >= max_per_outlet_total:
                continue
            
            selected_for_orientation.append(idx)
            outlet_counter[source] += 1
            topic_counter[topic] += 1
            
            if len(selected_for_orientation) == target_per_orientation:
                break
    
    # Final fallback: if still too few, relax outlet constraint too
    if len(selected_for_orientation) < target_per_orientation:
        remaining_needed = target_per_orientation - len(selected_for_orientation)
        
        remaining_pool = orientation_pool[
            ~orientation_pool.index.isin(selected_for_orientation)
        ]
        
        extra_indices = remaining_pool.head(remaining_needed).index.tolist()
        selected_for_orientation.extend(extra_indices)
        
        for idx in extra_indices:
            source = orientation_pool.loc[idx, "source"]
            topic = orientation_pool.loc[idx, "topics"]
            outlet_counter[source] += 1
            topic_counter[topic] += 1
    
    selected_rows.extend(selected_for_orientation)

validation_sample = length_filtered_df.loc[selected_rows].copy()

validation_sample = validation_sample.reset_index().rename(
    columns={"index": "original_index"}
)

validation_sample.shape

(45, 11)

In [41]:
# Political orientation balance
validation_sample["allsides_bias"].value_counts()

allsides_bias
From the Left      15
From the Center    15
From the Right     15
Name: count, dtype: int64

In [42]:
# Topic distribution in selected sample
validation_sample["topics"].value_counts()

topics
Election: Presidential         4
White House                    4
Terrorism                      4
Politics                       4
Supreme Court                  4
Violence in America            3
Healthcare                     2
World                          2
Polarization                   2
Foreign Policy                 1
Medicare                       1
Economy and Jobs               1
Cybersecurity                  1
National Defense               1
Elections                      1
Democratic Party               1
Middle East                    1
NSA                            1
Gun Control and Gun Rights     1
Immigration                    1
Media Bias                     1
Trade                          1
Fiscal Cliff, Budget - Debt    1
Abortion                       1
US Congress                    1
Name: count, dtype: int64

In [43]:
# Outlet distribution in selected sample
validation_sample["source"].value_counts()

source
Politico                     3
CNN (Web News)               3
Fox News                     3
Reuters                      3
USA TODAY                    3
BBC News                     3
Christian Science Monitor    3
Washington Times             3
CBN                          2
Business Insider             2
Los Angeles Times            2
NBCNews.com                  2
Townhall                     2
New York Times - News        2
HuffPost                     1
The Guardian                 1
Daily Beast                  1
The Hill                     1
TheBlaze.com                 1
Fox News Editorial           1
Newsmax                      1
David Brooks                 1
American Spectator           1
Name: count, dtype: int64

In [44]:
# Length summary of selected sample
validation_sample["word_count"].describe()

count     45.000000
mean     819.511111
std        2.958979
min      813.000000
25%      818.000000
50%      820.000000
75%      822.000000
max      825.000000
Name: word_count, dtype: float64

In [45]:
# Compare full pool, length-filtered pool, and validation sample length
length_comparison = pd.DataFrame({
    "Corpus": ["Full sampling pool", "Length-filtered pool", "Validation sample"],
    "Mean word count": [
        sampling_df["word_count"].mean(),
        length_filtered_df["word_count"].mean(),
        validation_sample["word_count"].mean()
    ],
    "Min word count": [
        sampling_df["word_count"].min(),
        length_filtered_df["word_count"].min(),
        validation_sample["word_count"].min()
    ],
    "Max word count": [
        sampling_df["word_count"].max(),
        length_filtered_df["word_count"].max(),
        validation_sample["word_count"].max()
    ]
})

length_comparison

,Corpus,Mean word count,Min word count,Max word count
0,Full sampling pool,819.816677,1,69312
1,Length-filtered pool,821.027346,700,950
2,Validation sample,819.511111,813,825


In [46]:
validation_sample[
    [
        "original_index",
        "allsides_bias",
        "source",
        "topics",
        "title",
        "word_count"
    ]
].sort_values(["allsides_bias", "topics", "source"])

,original_index,allsides_bias,source,topics,title,word_count
16,2901,From the Center,BBC News,Cybersecurity,WannaCry ransomware cyber-attacks slow but fea...,820
27,3434,From the Center,Business Insider,Democratic Party,,824
19,3452,From the Center,Christian Science Monitor,Election: Presidential,Trump rides rural rebellion to stunning victory,818
25,3668,From the Center,Christian Science Monitor,Election: Presidential,How much hope for third-party presidential can...,823
26,971,From the Center,Reuters,Elections,Trump-backed Republican clings to narrow lead ...,823
18,2536,From the Center,BBC News,National Defense,Trump warns N Korea that US military is 'locke...,821
20,995,From the Center,Reuters,Politics,Trump urges attorney general to end Russia pro...,818
28,1317,From the Center,Reuters,Politics,"Trump pardons Dinesh D'Souza, says lifestyle m...",815
17,628,From the Center,BBC News,Supreme Court,Brett Kavanaugh confirmation: Victory for Trum...,819
15,647,From the Center,The Hill,Supreme Court,Grassley: No corroboration of Kavanaugh accuse...,820


In [47]:
validation_sample.to_csv(
    "validation_sample_45_articles.csv",
    index=False
)

validation_sample.to_json(
    "validation_sample_45_articles.json",
    orient="records",
    lines=True
)

In [48]:
from pathlib import Path
import re

# Create output folder
output_dir = Path("validation_sample_txt")
output_dir.mkdir(exist_ok=True)

def clean_filename(text):
    """
    Create safe filenames by removing characters that can cause problems.
    """
    text = str(text)
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"\s+", "_", text.strip())
    return text[:80]

# Save each article as a separate .txt file
for i, row in validation_sample.iterrows():
    article_id = row["original_index"]
    orientation = clean_filename(row["allsides_bias"])
    source = clean_filename(row["source"])
    topic = clean_filename(row["topics"])
    
    filename = f"{i+1:02d}_{orientation}_{source}_{topic}_{article_id}.txt"
    filepath = output_dir / filename
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(str(row["content"]))

print(f"Saved {len(validation_sample)} articles to folder: {output_dir}")

Saved 45 articles to folder: validation_sample_txt


In [49]:
from pathlib import Path

# Create folder for anonymized article text files
output_dir = Path("validation_sample_txt")
output_dir.mkdir(exist_ok=True)

# Create neutral article IDs
validation_sample = validation_sample.copy()
validation_sample["article_id"] = [
    f"article_{i:02d}" for i in range(1, len(validation_sample) + 1)
]

# Save each article as a neutral .txt file
for _, row in validation_sample.iterrows():
    filename = f"{row['article_id']}.txt"
    filepath = output_dir / filename
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(str(row["content"]))

# Save metadata separately
metadata_cols = [
    "article_id",
    "original_index",
    "allsides_bias",
    "source",
    "topics",
    "title",
    "word_count"
]

validation_sample[metadata_cols].to_csv(
    "validation_sample_metadata.csv",
    index=False
)

print(f"Saved {len(validation_sample)} anonymized articles to: {output_dir}")
print("Saved metadata file as: validation_sample_metadata.csv")

Saved 45 anonymized articles to: validation_sample_txt
Saved metadata file as: validation_sample_metadata.csv


I selected the validation set using a stratified and constrained sampling strategy, because the goal was not to create a purely random subset, but a subset that is suitable for the thesis comparison.

Since political orientation is the main variable in the thesis, I first balanced the sample across the three orientations: 15 left-leaning articles, 15 center articles, and 15 right-leaning articles. This prevents the validation results from being dominated by the larger orientation groups in the full corpus.

I also controlled for article length by selecting articles between 700 and 950 words, with the final sample centered very closely around the corpus mean length. This was important because the number of argumentative units may depend on article length; very short or very long texts could distort the annotation process.

In addition, I limited the number of articles from the same topic and the same outlet. No topic appears more than four times, and no outlet appears more than three times. This was done to make the subset more diverse and to reduce the risk that the validation results reflect only one news topic or one outlet’s writing style.

The resulting subset is therefore appropriate for the thesis because it is balanced across political orientation, controlled for length, and diverse across topics and sources. It is not meant to be fully proportional to the entire corpus, but rather analytically representative of the dimensions that are most relevant for validating the annotation and analysis of argumentative units.

I chose a maximum of four articles per topic to ensure topic diversity within the limited 45-article validation set. This prevents the subset from being overly shaped by one dominant topic, while still allowing major recurring topics in the corpus to be represented. Since four articles correspond to less than 10% of the validation sample, this threshold provides a reasonable balance between including frequent topics and maintaining thematic variety.